# BGIS Feasibility Assessment

This notebook runs the BGIS feasibility analysis for blue-green infrastructure (BGI) siting.
It takes land use/land cover data and a BGI catalog, checks which BGI types are suitable for
each land parcel, and outputs the results as a GeoPackage.

Originally developed as a standalone Python script by Kim Wang at Wageningen University & Research.
Converted to a Jupyter notebook and adapted for NaaVRE by Sven Tesselaar (UvA) as part of a BSc thesis.

## Workflow components
1. **Data Loader** - reads and parses the input files
2. **Land Cover Mapper** - translates land use types to installation layers
3. **Feasibility Checker** - checks installation layer and area compatibility
4. **Output Generator** - assembles results and writes the feasibility output file
5. **Optimizer** - selects one BGI per parcel using binary integer programming (PuLP)

## Files and paths
A `data/` folder next to this notebook holds the example inputs
(`example_amsterdam_neighborhood.gpkg`, `bgi_catalog_v1.xlsx`). This is useful for
running the notebook locally in JupyterLab and for keeping the repo self-contained.

For containerised workflow runs in NaaVRE, the cell containers cannot see the
JupyterLab filesystem. Input files must be in Cloud Storage
(`/home/jovyan/Cloud Storage/naa-vre-user-data/`), which is the default in Cell 1.
Upload the same files from `data/` to Cloud Storage before running the workflow.

## NaaVRE conventions
- `param_` variables: defined in the top params cell, editable by users at workflow runtime
- `/tmp/data/`: intermediate file storage between containerised cells
- Outputs from earlier cells connect to inputs of later cells; NaaVRE handles execution order


In [1]:
# params
# this cell defines all workflow parameters
# NaaVRE will expose these in the Run dialog at workflow runtime

# data loader parameters
param_input_gpkg_path = "/home/jovyan/Cloud Storage/naa-vre-public/vl-bgis-urban-planning/example_amsterdam_neighborhood.gpkg"
param_input_catalog_path = "/home/jovyan/Cloud Storage/naa-vre-public/vl-bgis-urban-planning/bgi_catalog_v1.xlsx"

# land cover mapper parameter
param_landcover_dict_path = 'default'

# output generator parameter
param_output_gpkg_path = '/home/jovyan/Cloud Storage/naa-vre-user-data/testing_standalone.gpkg'

# optimizer parameters
param_cooling_goal = 1.0        # goal expressed as average deg C cooled per m2
param_retention_goal = 0.2      # goal expressed as average depth of storage (m) per m2
param_penalty_cooling = 1.0     # penalty weight for unmet cooling target
param_penalty_retention = 1.0   # penalty weight for unmet retention target
                                 # weights are relative: 1:1 means equally penalised
param_optimization_output_path = '/home/jovyan/Cloud Storage/naa-vre-user-data/bgis_optimization_result.gpkg'


In [1]:
# data_loader_BGIS

import geopandas as gpd
import pandas as pd
import ast
import numpy as np
import json
import os
import pickle

##################################################################################
# functions for loading text correctly
##################################################################################

def safe_parse(cell):
    """Correctly read excel sheet values.
    Handles missing values, booleans, JSON strings, literal Python objects,
    plain strings, and dictionaries."""
    if pd.isna(cell):
        return None
    if isinstance(cell, str):
        cleaned_string = cell.strip()
        if cleaned_string.lower() in ['true', 'false']:
            return cleaned_string.lower() == 'true'
        try:
            parsed = json.loads(cleaned_string)
        except Exception:
            parsed = None
        if parsed is None:
            try:
                parsed = ast.literal_eval(cleaned_string)
            except Exception:
                parsed = cleaned_string
        cell = parsed
    if isinstance(cell, str):
        return cell.strip()
    if isinstance(cell, dict):
        return tuple(sorted(cell.items()))
    return cell


def check_list(value):
    """For when strings need to be in list format."""
    if value is None:
        return []
    if isinstance(value, list):
        return value
    return [value]

##################################################################################
# load shapefile here
# example given is a small section of Amsterdam centrum
# (a few blocks from Keizersgracht - Prinsengracht)
# taken from Amsterdam Gemeente data (BGT + 3DBAG flat roof dataset)
##################################################################################

LULC_GDF = gpd.read_file(param_input_gpkg_path)
LULC_GDF['type'] = LULC_GDF['type'].apply(safe_parse)
gdf = LULC_GDF.copy()

if 'area_m2' not in gdf.columns:
    gdf['area_m2'] = gdf.geometry.area

##################################################################################
# load in excel sheet with information on BGI technology here
# can add new categories or modify existing easily
##################################################################################

BGI_SHEET = pd.read_excel(param_input_catalog_path, sheet_name=0)

load_as_string_headers = [
    'install_layer', 'geometry', 'soil_type', 'adaptive_siting',
    'vegetation', 'modular', 'maintenance_requirement', 'permit'
]

bgi_catalog = {}
for index, row in BGI_SHEET.iterrows():
    bgi_name = row['bgi_name']
    entry = {}
    for column in BGI_SHEET.columns:
        if column in load_as_string_headers:
            entry[column] = safe_parse(row.get(column))
        else:
            entry[column] = row.get(column)
    bgi_catalog[bgi_name] = entry

##################################################################################
# save outputs to /tmp/data/ for the next cells
##################################################################################

os.makedirs('/tmp/data', exist_ok=True)
gdf.to_file('/tmp/data/gdf.gpkg', driver='GPKG')
with open('/tmp/data/bgi_catalog.pkl', 'wb') as f:
    pickle.dump(bgi_catalog, f)

gdf_path = '/tmp/data/gdf.gpkg'
catalog_path = '/tmp/data/bgi_catalog.pkl'

print(f'Loaded {len(gdf)} parcels and {len(bgi_catalog)} BGI types.')


NameError: name 'param_input_gpkg_path' is not defined

In [3]:
# land_cover_mapper

import geopandas as gpd
import json
import os

##################################################################################
# parameter - optional custom landcover dictionary as a JSON file
# if left as 'default', the Amsterdam dictionary below is used
# to use a custom mapping, upload a JSON file to Cloud Storage and set this
# parameter to the path, e.g.:
# /home/jovyan/Cloud Storage/naa-vre-user-data/my_city_mapping.json
#
# the JSON file should look like:
# {"local_type_name": "installation_layer", ...}
# e.g.: {"rijbaan_lokale_weg": "pavement", "groenvoorziening": "ground"}
##################################################################################

##################################################################################
# load data from previous cell
##################################################################################

gdf = gpd.read_file(gdf_path)

##################################################################################
# default Amsterdam landcover dictionary
# maps Amsterdam BGT/3DBAG type labels to generic installation layers
# used when param_landcover_dict_path is 'default'
##################################################################################

default_landcover_dictionary = {
    'baan_voor_vliegverkeer': 'pavement',
    'bassin': 'feature',
    'berm': 'ground',
    'erf': 'ground',
    'fietspad': 'pavement',
    'flat roof': 'flatroof',
    'gesloten_verharding': 'ground',
    'grasland_overig': 'ground',
    'groenvoorziening': 'ground',
    'half_verhard': 'ground',
    'inrit': 'none',
    'kademuur_v': 'quaywall',
    'lage_trafo': 'infrastructure',
    'muur_v': 'wall',
    'oever_slootkant': 'bank',
    'onverhard': 'ground',
    'open_loods': 'aviary',
    'open_verharding': 'ground',
    'ov-baan': 'pavement',
    'pand': 'angledroof',
    'parkeervlak': 'pavement',
    'perron': 'none',
    'rijbaan_lokale_weg': 'pavement',
    'rijbaan_regionale_weg': 'pavement',
    'sluis': 'infrastructure',
    'struiken': 'ground',
    'verkeerseiland': 'ground',
    'voetgangersgebied': 'pavement',
    'voetpad': 'pavement',
    'voetpad_op_trap': 'stairs',
    'waterloop': 'water',
    'woonerf': 'ground',
    'zand': 'ground'
}

##################################################################################
# load custom dictionary from JSON if provided, otherwise use default
##################################################################################

if param_landcover_dict_path != 'default' and os.path.exists(param_landcover_dict_path):
    with open(param_landcover_dict_path, 'r') as f:
        landcover_dictionary = json.load(f)
    print(f'Loaded custom landcover dictionary from {param_landcover_dict_path}')
else:
    landcover_dictionary = default_landcover_dictionary
    print('Using default Amsterdam landcover dictionary')

# translation occurs here, adding another column to gdf
gdf['install_layer'] = gdf['type'].str.lower().map(landcover_dictionary).fillna('none')

##################################################################################
# save output to /tmp/data/ for the next cells
##################################################################################

gdf.to_file('/tmp/data/mapped_gdf.gpkg', driver='GPKG')
mapped_gdf_path = '/tmp/data/mapped_gdf.gpkg'

print(f'Mapped {gdf["install_layer"].nunique()} unique installation layers.')


Using default Amsterdam landcover dictionary
Mapped 9 unique installation layers.


In [4]:
# feasibility_checker

import geopandas as gpd
import pandas as pd
import numpy as np
import pickle

##################################################################################
# load data from previous cells
##################################################################################

gdf = gpd.read_file(mapped_gdf_path)

with open(catalog_path, 'rb') as f:
    bgi_catalog = pickle.load(f)

##################################################################################
# creating masks for various condition checks
##################################################################################

def check_install_layer(gdf, catalog):
    """Check if the installation layer of the land matches one of the BGIs.
    Handles both list and string values for install_layer."""
    catalog_layers = {}
    for name, entry in catalog.items():
        val = entry.get('install_layer')
        if val is None:
            layers = set()
        elif isinstance(val, list):
            layers = set(val)
        else:
            layers = {val}
        catalog_layers[name] = layers
    series = gdf['install_layer']
    return pd.DataFrame({
        name: series.isin(layers)
        for name, layers in catalog_layers.items()
    })


def check_area(gdf, catalog):
    """Check if the land polygon has the minimum or maximum area
    as specified by BGI characteristic."""
    catalog_min_area = {
        name: float(entry['area_min']) if pd.notna(entry['area_min']) else 0
        for name, entry in catalog.items()
    }
    catalog_max_area = {
        name: float(entry['area_max']) if pd.notna(entry['area_max']) else np.inf
        for name, entry in catalog.items()
    }
    series = gdf['area_m2']
    return pd.DataFrame({
        name: (series >= catalog_min_area[name]) & (series <= catalog_max_area[name])
        for name in catalog.keys()
    })

##################################################################################
# run checks and combine masks
##################################################################################

lulc_mask = check_install_layer(gdf, bgi_catalog)
area_mask = check_area(gdf, bgi_catalog)
final_mask = lulc_mask & area_mask

##################################################################################
# save output to /tmp/data/ for the next cell
##################################################################################

final_mask.to_pickle('/tmp/data/final_mask.pkl')
mask_path = '/tmp/data/final_mask.pkl'

print(f'Feasibility checks complete. {final_mask.any().sum()} BGI types have at least one suitable parcel.')

Feasibility checks complete. 9 BGI types have at least one suitable parcel.


In [5]:
# output_generator

import geopandas as gpd
import pandas as pd
import pickle

##################################################################################
# load data from previous cells
##################################################################################

gdf = gpd.read_file(mapped_gdf_path)

final_mask = pd.read_pickle(mask_path)

with open(catalog_path, 'rb') as f:
    bgi_catalog = pickle.load(f)

##################################################################################
# creating final output
# making the mask go from boolean to a readable result
##################################################################################

gdf['suitable_bgis'] = final_mask.apply(
    lambda row: [bgi for bgi, valid in row.items() if valid],
    axis=1
)

# add binary matrix of 0/1s for easier visualization in QGIS
bgi_binary_matrix = pd.DataFrame(
    0,
    index=gdf.index,
    columns=list(bgi_catalog.keys())
)

for idx, bgis in gdf['suitable_bgis'].items():
    if bgis:
        bgi_binary_matrix.loc[idx, bgis] = 1

output_gdf = pd.concat([gdf, bgi_binary_matrix], axis=1)

##################################################################################
# save file to cloud storage
##################################################################################

output_gdf.to_file(param_output_gpkg_path, layer='feasibility', driver='GPKG')

##################################################################################
# expose output path so the optimizer cell can receive it as an NaaVRE input
##################################################################################

feasibility_output_path = param_output_gpkg_path

print(f'Output saved to {param_output_gpkg_path} with {len(output_gdf)} parcels.')


Output saved to /home/jovyan/Cloud Storage/naa-vre-user-data/testing_standalone.gpkg with 2071 parcels.


## Optimizer

Selects one BGI per parcel using binary integer programming (PuLP).
Takes the feasibility output from the Output Generator and optimises toward
user-defined cooling and water retention targets using a soft-goal formulation:
it minimises the penalty-weighted shortfall from each target rather than
rejecting runs that fall short.

Originally developed as `optimization_draft` by Kim Wang at Wageningen University & Research.
Adapted for NaaVRE by Sven Tesselaar (UvA).

### Inputs (from earlier cells)
- `feasibility_output_path`: path to feasibility GeoPackage written by the Output Generator
- `catalog_path`: path to the BGI catalog pickle written by the Data Loader

### Outputs
- Optimization GeoPackage saved to `param_optimization_output_path`
- PNG map saved alongside the GeoPackage for quick inspection in JupyterLab


In [6]:
# optimizer

#!pip install pulp
''' you may need to install pulp yourself, it is not always automatically combined with other python packages'''

import geopandas as gpd
import pyogrio
import pandas as pd
import ast
import numpy as np
from pulp import LpProblem, LpMaximize, LpMinimize, LpVariable, lpSum, LpBinary, LpStatus, PULP_CBC_CMD
import json
import time
import pickle
import matplotlib
matplotlib.use('Agg')  # non-interactive backend required in containers (no display available)
import matplotlib.pyplot as plt

##################################################################################
# load data from previous cells
# feasibility_output_path comes from output_generator (Cell 4)
# catalog_path comes from data_loader (Cell 1)
# strip extra quotes that NaaVRE may add when passing string inputs between cells
##################################################################################

feasibility_output_path = feasibility_output_path.strip('"')
catalog_path = catalog_path.strip('"')

################################################################
#             parsing to load catalog from excel               #
################################################################

def safe_parse(cell):
    """ for reading the excel sheet """
    
    if pd.isna(cell):                                       # case: missing value
        return None                                       
    
    if isinstance(cell, str):                               # case: is string
        cleaned_string = cell.strip()                     
        
        if cleaned_string.lower() in ["true", "false"]:     # case: Boolean
            return cleaned_string.lower() == "true"       

        try:                                              
            parsed = json.loads(cleaned_string)             # attempt JSON parsing
        except Exception:                                 
            parsed = None

        if parsed is None:
            try:
                parsed = ast.literal_eval(cleaned_string)   # attempt literal parsing
            except Exception:
                parsed = cleaned_string
        cell = parsed

    if isinstance(cell, str):                               # case: string 
        return cell.strip()

    if isinstance(cell, dict):                              # case: dictionary
        return tuple(sorted(cell.items()))
        
    return cell 


def check_list(value):
    ''' when strings need to be in lists ''' 
    if value is None: 
        return []
    if isinstance(value, list):
        return value
    return [value]

################################################################
#             load in BGI_CATALOG dictionary                   #
#             loaded from pickle written by data_loader        #
################################################################

with open(catalog_path, 'rb') as f:
    BGI_CATALOG = pickle.load(f)

################################################################
#         convert catalog to dictionary: bgi_df                #
#         convert dataframe columns to dictionaries            #
################################################################

bgi_df = pd.DataFrame.from_dict(BGI_CATALOG, orient="index")
bgi_cooling = {bgi: BGI_CATALOG[bgi]['temperature_reduction'] for bgi in BGI_CATALOG}
bgi_retention = {bgi: BGI_CATALOG[bgi]['water_storage_volume'] for bgi in BGI_CATALOG}
bgi_capex = {bgi: BGI_CATALOG[bgi]['CAPEX'] for bgi in BGI_CATALOG}

################################################################
#                read in feasibility result                    #
################################################################

FEASIBILITY_RESULT = gpd.read_file(feasibility_output_path)
gdf = FEASIBILITY_RESULT.copy()
gdf['suitable_bgis'] = gdf['suitable_bgis'].apply(safe_parse)

################################################################
#   select MAX area of BGI application OR MAX area of cell     #
################################################################

area_cap = {}
for row in gdf.itertuples():
    idx = row.Index
    if not isinstance(row.suitable_bgis, list):
        continue
    for bgi in row.suitable_bgis:
        if bgi not in BGI_CATALOG:
            raise KeyError(f"missing BGI key, {bgi}")
            continue
        cap = min(
            row.area_m2,
            BGI_CATALOG[bgi].get('area_max', float('inf'))
        )
        if cap > 0:
            area_cap[(idx, bgi)] = cap

################################################################
#                 theoretical maximum goals                    #
################################################################

total_area = gdf['area_m2'].sum()
max_cooling = {}; max_storage = {}
for (idx, bgi), area in area_cap.items():
    cooling_value = area * BGI_CATALOG[bgi]['temperature_reduction']
    storage_value = area * BGI_CATALOG[bgi]['water_storage_volume']
    if idx not in max_cooling:
        max_cooling[idx] = cooling_value
    else:
        max_cooling[idx] = max(max_cooling[idx], cooling_value)
    if idx not in max_storage:
        max_storage[idx] = storage_value
    else: 
        max_storage[idx] = max(max_storage[idx], storage_value)
    
total_theoretical_cooling = sum(max_cooling.values())
total_theoretical_storage = sum(max_storage.values())
print(f"max cooling in C per m2: {total_theoretical_cooling/total_area:.3f}")
print(f"max storage in depth m: {total_theoretical_storage/total_area:.3f}")
print(f"total area in m2: {total_area:.2f}")

################################################################
#  optimization function where goals are soft constraint       #
#  returned solution is as close to both goals as possible     #
#  no budget constraint in place                               #
################################################################

def softgoal_hardbudget(avg_cooling_target, avg_retention_target, penalty_cooling=1, penalty_retention=1):
    start_time = time.time()
    cooling_target = avg_cooling_target * total_area
    retention_target = avg_retention_target * total_area
    variables = {}
    for (idx, bgi) in area_cap:
            variables[(idx, bgi)] = LpVariable(f"x_{idx}_{bgi}", cat='Binary')

    slack_cooling = LpVariable("slack_cooling", lowBound=0)
    slack_retention = LpVariable("slack_retention", lowBound=0)
    
    prob = LpProblem("Goal_Optimization", LpMinimize)

    # select 1 BGI per cell 
    for idx in gdf.index:
        prob += lpSum(
            variables[(idx, bgi)]
            for bgi in (gdf.at[idx, 'suitable_bgis'] or [])
            if (idx, bgi) in variables
        ) <= 1
    
    # temperature target 
    prob += lpSum(
        variables[(idx, bgi)] * area_cap[(idx, bgi)] * bgi_cooling[bgi]
        for (idx, bgi) in variables
    ) + slack_cooling >= cooling_target

    # retention target
    prob += lpSum(
        variables[(idx, bgi)] * area_cap[(idx, bgi)] * bgi_retention[bgi]
        for (idx, bgi) in variables
    ) + slack_retention >= retention_target
     
    prob += penalty_cooling * slack_cooling + penalty_retention * slack_retention    
    
    # try CBC solver first, fall back to default if not available in container
    try:
        prob.solve(PULP_CBC_CMD(msg=1, timeLimit=120))
    except Exception:
        prob.solve()
    runtime = time.time() - start_time

    gdf_result = gdf.copy()
    gdf_result['selected_bgi'] = None
    gdf_result['selected_bgi_area'] = 0.0
    gdf_result['selected_capex'] = 0.0
    
    for (idx, bgi), var in variables.items():
        if var.value() is not None and var.value() > 0.9:
            gdf_result.at[idx, 'selected_bgi'] = bgi
            gdf_result.at[idx, 'selected_bgi_area'] = area_cap[(idx,bgi)]
            gdf_result.at[idx, 'selected_capex'] = bgi_capex.get(bgi, 0) * area_cap[(idx, bgi)]

    print(f"status: {LpStatus[prob.status]} in {runtime:.1f} seconds")
    print(f"Cooling shortfall: {slack_cooling.value()}")
    print(f"Retention shortfall: {slack_retention.value()}")
    print()
    
    return(gdf_result)

################################################################
#  run optimization function                                   #
#  !!!!!!! edit your own goals and penalty weights here        #
#  !!!!!!! edit your own save file name etc here               #
################################################################

solutions_gdf = softgoal_hardbudget(
    avg_cooling_target=param_cooling_goal,
    avg_retention_target=param_retention_goal,
    penalty_cooling=param_penalty_cooling,
    penalty_retention=param_penalty_retention
)

solutions_gdf.to_file(param_optimization_output_path, layer="optimization_result", driver="GPKG")
print(f"Optimization result saved to {param_optimization_output_path}")

################################################################
#  save static PNG map to cloud storage                        #
#  for quick inspection in JupyterLab without downloading      #
#  matplotlib.use('Agg') at top ensures no display needed      #
################################################################

fig, ax = plt.subplots(1, 1, figsize=(10, 10))
solutions_gdf.plot(
    column='selected_bgi',
    ax=ax,
    legend=True,
    missing_kwds={'color': 'lightgrey', 'label': 'No BGI selected'},
    cmap='tab10'
)
ax.set_title('Selected BGI per parcel')
ax.set_axis_off()

optimization_map_path = param_optimization_output_path.replace('.gpkg', '_map.png')
fig.savefig(optimization_map_path, dpi=150, bbox_inches='tight')
plt.close(fig)

print(f"Map saved to {optimization_map_path}")

max cooling in C per m2: 1.947
max storage in depth m: 0.311
total area in m2: 159236.69
GLPSOL--GLPK LP/MIP Solver 5.0
Parameter(s) specified in the command line:
 --cpxlp /tmp/90f883bde49d412880414e6289a29bc7-pulp.lp -o /tmp/90f883bde49d412880414e6289a29bc7-pulp.sol
Reading problem data from '/tmp/90f883bde49d412880414e6289a29bc7-pulp.lp'...
2074 rows, 1933 columns, 6024 non-zeros
1930 integer variables, all of which are binary
6177 lines were read
GLPK Integer Optimizer 5.0
2074 rows, 1933 columns, 6024 non-zeros
1930 integer variables, all of which are binary
Preprocessing...
579 rows, 1932 columns, 4529 non-zeros
1930 integer variables, all of which are binary
Scaling...
 A: min|aij| =  2.020e-01  max|aij| =  1.642e+04  ratio =  8.126e+04
GM: min|aij| =  2.812e-01  max|aij| =  3.557e+00  ratio =  1.265e+01
EQ: min|aij| =  7.906e-02  max|aij| =  1.000e+00  ratio =  1.265e+01
2N: min|aij| =  3.760e-02  max|aij| =  1.197e+00  ratio =  3.184e+01
Constructing initial basis...
Size of t